# ICE Outcomes: Data Exploration & Cleaning

### Data Pre-Cleaning and Sampling
---

Before performing data cleaning and preprocessing in Python, we first used Google Cloud Platform and BigQuery to organize and pre-clean the raw immigration datasets.

The raw dataset was structured as separate tables for each year (2012–2023) and by category, including Arrests, Decisions, Detentions, and Removals. For each year, we combined these four tables into a single master table (e.g., master_50k_12, master_50k_2013, etc.) using BigQuery joins and queries. This allowed us to consolidate all relevant information for each case into one table per year.

After creating the yearly master tables, we performed sampling in BigQuery. We used a simple random sampling method to select 50,000 unqiue identifiers and their records per year. During sampling, we filtered for records where the target variable (`final_order`, Yes/No) was not NULL to ensure the data would be usable for later analysis and modeling.

After this preprocessing and sampling step in BigQuery, we exported the sampled data and continued further data cleaning, preprocessing, and analysis in Python.

### 2012 - 2023 deportation data
--- 

In [17]:
# import modules
import pandas as pd
from google.cloud import bigquery

# connet to BigQuery
client = bigquery.Client(project="ice-data-project")

#### 2012 Deportation Data

In [18]:
# 50k SRS 2012 dataset
query = """
SELECT *
FROM `ice-data-project.ice_data_clean.master_50k_12`
"""

deportations_12 = client.query(query).to_dataframe()

# SHOW ALL COLUMNS IN DATAFRAME
record_columns = ['arrests', 'decisions', 'detentions', 'removals']
for col in record_columns:
    # 1. Convert any missing records (NaN) into empty dictionaries to prevent crashes
    deportations_12[col] = deportations_12[col].apply(lambda x: x if isinstance(x, dict) else {})
    
    # 2. Flatten the dictionary into its own temporary DataFrame
    # Adding a prefix ensures we know where the data came from (e.g., 'detentions_Case ID')
    deportations_12_flat = pd.json_normalize(deportations_12[col]).add_prefix(f'{col}_')
    
    # 3. Attach the newly flattened columns back to the main DataFrame
    deportations_12 = pd.concat([deportations_12, deportations_12_flat], axis=1)
    
    # 4. Drop the original nested column to keep the DataFrame clean
    deportations_12 = deportations_12.drop(columns=[col])
print("\nData successfully flattened!")


/Library/Frameworks/Python.framework/Versions/3.14/lib/python3.14/site-packages/google/cloud/bigquery/table.py:1994: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(



Data successfully flattened!


In [19]:
deportations_12.head()

,Anonymized Identifier,arrests_Apprehension Date,arrests_Apprehension Method,arrests_Arrest Created By,arrests_Case ID,arrests_Subject ID,arrests_Alien File Number,arrests_Anonymized Identifier,decisions_RCA_AOR,decisions_RCA_DCO,...,removals_MSC Conviction Date,removals_MSC Criminal Charge Status,removals_Case Threat Level,removals_Processing Disposition Code,removals_Processing Disposition,removals_Current Program,removals_Apprehension Date,removals_Charge Section Code,removals_Charge Code,removals_Anonymized Identifer
0,d0451cac01101d02e3b5c236e6b7432e5eff2148,2012-02-29,CAP Local Incarceration,(b)(6)(b)(7)(c),(b)(6)(b)(7)(c),(b)(6)(b)(7)(c),(b)(6)(b)(7)(c),d0451cac01101d02e3b5c236e6b7432e5eff2148,None,None,...,NaT,None,None,None,None,None,NaT,None,None,None
1,722038e8783f0111938ba418485b72719130bb42,2012-06-12,CAP Local Incarceration,(b)(6)(b)(7)(c),(b)(6)(b)(7)(c),(b)(6)(b)(7)(c),(b)(6)(b)(7)(c),722038e8783f0111938ba418485b72719130bb42,None,None,...,NaT,None,None,None,None,None,NaT,None,None,None
2,f196886fb97872592bce7642b1c1ea79722672b5,2012-08-23,CAP Local Incarceration,(b)(6)(b)(7)(c),None,(b)(6)(b)(7)(c),(b)(6)(b)(7)(c),f196886fb97872592bce7642b1c1ea79722672b5,None,None,...,NaT,None,None,None,None,None,NaT,None,None,None
3,764bb6cc37b909a941aeebe9ae5f813bfe450c87,NaT,None,None,None,None,None,None,None,None,...,NaT,None,None,None,None,None,NaT,None,None,None
4,c269a353ec849af9c41c95409e30fe9b960889c7,NaT,None,None,None,None,None,None,None,None,...,NaT,None,None,None,None,None,NaT,None,None,None


In [20]:
# List all columns
print("Columns in the dataframe:")
print(deportations_12.columns.tolist())

# Number of rows and columns
print(f"\nNumber of rows: {deportations_12.shape[0]}")
print(f"Number of columns: {deportations_12.shape[1]}")

# Number of nulls per column, sorted descending
null_counts = deportations_12.isnull().sum().sort_values(ascending=False)
print("\nTop columns by number of null values:")
print(null_counts[null_counts > 100000])  # only show columns with at least 1 null

Columns in the dataframe:
['Anonymized Identifier', 'arrests_Apprehension Date', 'arrests_Apprehension Method', 'arrests_Arrest Created By', 'arrests_Case ID', 'arrests_Subject ID', 'arrests_Alien File Number', 'arrests_Anonymized Identifier', 'decisions_RCA_AOR', 'decisions_RCA_DCO', 'decisions_A_NUMBER', 'decisions_SUBJ_ID', 'decisions_LAST_NAME', 'decisions_FIRST_NAME', 'decisions_ALERT_CODE', 'decisions_ACTIVE_INACTIVE', 'decisions_SUBMISSION_DATE', 'decisions_STATUS_CODE', 'decisions_RISK_TO_PUBLIC_SAFETY', 'decisions_RISK_OF_FLIGHT', 'decisions_SPECIAL_VULNERABILITY', 'decisions_RCA_CASE_NUMBER', 'decisions_CASE_CAT_AT_RCA_DECISION', 'decisions_FO_AT_RCA_DECISION', 'decisions_FO_DATE_AT_RCA_DECISION', 'decisions_REMOVAL_LIKELY_AT_RCA_DECISION', 'decisions_MAN_DET_PER_STAT_ALLEG', 'decisions_RCA_DECISION_TYPE', 'decisions_RCA_RECOMMENDATION', 'decisions_RCA_BOND_RECOMMENDATION', 'decisions_OFFICER_ID', 'decisions_OFFICER_AGREE_DISAGREE', 'decisions_SUPERVISOR_ID', 'decisions_SUPER

In [21]:
# Count unique values in the 'Anonymized Identifier' column
unique_ids = deportations_12['Anonymized Identifier'].nunique()
print(f"Number of unique Anonymized Identifiers: {unique_ids}")

# Count occurrences of each identifier
id_counts = deportations_12['Anonymized Identifier'].value_counts()
print(id_counts.head(10))  # show top 10 most frequent

Number of unique Anonymized Identifiers: 49999
Anonymized Identifier
0c3b60a99bb8cde4842ea33406925bd86415dc6e    324
4b24ab94116b4f9dc2526df4b8bb390f802f5c2d    240
f7b6851db74ee0632fdda3b48d7122839a694bbc    132
5e7b18264fd9f0b8b94565b960deb8f134395915    120
64ed1e275383b5bcaaaf19e75de28d4af30a9769    110
77cea3587cfcbc081e74162fc14579880d7d4161    108
a8a126df5f7304f860f817a5c97d1e81bc232a94    105
03345c5ab1d75f0f544610fd0bb84868058c2493    104
c4b538597d43914f82e1bc54c63e8dc936b392eb     90
24501b2bbbb8fa92e0c5920b123f9eb52918cc61     84
Name: count, dtype: int64


#### 2013 Deportation Data

#### 2014 Deportation Data

#### 2015 Deportation Data

#### 2016 Deportation Data

#### 2017 Deportation Data

#### 2018 Deportation Data

#### 2019 Deportation Data

#### 2020 Deportation Data

#### 2021 Deportation Data

#### 2022 Deportation Data

### 2023 - 2025 deportation data
--- 

#### 2023 Deportation Data

#### 2024 Deportation Data

#### 2025 Deportation Data